# Agent Class Structure: Practice Exercise

In this exercise, you will implement the core class structure for a ReAct agent. The tools and testing code are provided - your task is to build the agent class that manages conversations and calls the LLM.

**What you'll implement:**
- The `__init__` method to initialize the agent with a system prompt
- The `__call__` method to handle user messages and manage conversation history
- The `execute` method to call the OpenAI API

**Estimated time:** 10-15 minutes

In [2]:
from openai import OpenAI
import os
import re

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ


if not IN_COLAB:
  from dotenv import load_dotenv
  load_dotenv()
  # Verify OpenAI API key is set
  if not os.getenv("OPENAI_API_KEY"):
    ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    OPEN_AI_API_KEY=os.getenv("OPENAI_API_KEY")
    print("OpenAI API key found. Ready to proceed!")
else:
  from google.colab import userdata
  OPEN_AI_API_KEY=userdata.get('OPEN_AI_API_KEY')

## Setup

Run this cell to import all required libraries and configure the environment.

In [3]:
# Setup - run this cell first

client = OpenAI(api_key=OPEN_AI_API_KEY)
print("Setup complete!")


Setup complete!


## Context

You are building a **movie recommendation agent** that helps users find movies based on genre and get runtime information. The agent uses the ReAct pattern to reason through queries step by step.

**Your task:** Implement the `MovieAdvisorAgent` class that:
1. Stores a system prompt and maintains a conversation history (list of messages)
2. Allows the agent to be called like a function with a user message
3. Sends the conversation to the OpenAI API and returns the response

**Class structure requirements:**
- `__init__(self, system_prompt)`: Initialize with system prompt, create empty messages list, add system message if provided
- `__call__(self, message)`: Add user message to history, call execute(), add assistant response to history, return response
- `execute(self)`: Call OpenAI API with self.messages, return the response content

**API details:**
- Use `client.chat.completions.create()` with model `"gpt-4o-mini"` and temperature `0`
- Messages format: `{"role": "system"|"user"|"assistant", "content": "..."}`
- Response content is at: `completion.choices[0].message.content`

## Your Task: Implement the Agent Class

Complete the `MovieAdvisorAgent` class below. This class manages the conversation state and communicates with the LLM.

In [4]:
class MovieAdvisorAgent:
    """
    A ReAct agent that uses OpenAI to reason about movie recommendations.

    The agent maintains a conversation history and can be called directly
    with user messages. It uses the OpenAI chat completions API to generate
    responses based on the full conversation context.

    Attributes:
        system_prompt (str): The system instructions that define agent behavior
        messages (list): The conversation history as a list of message dicts
    """

    def __init__(self, system_prompt: str = ""):
        """
        Initialize the agent with an optional system prompt.

        Args:
            system_prompt: Instructions that define how the agent should behave.
                          If provided, it should be added as the first message
                          with role "system".

        The method should:
        1. Store the system_prompt as an instance attribute
        2. Initialize self.messages as an empty list
        3. If system_prompt is not empty, append a system message to self.messages
        """
        # TODO: Implement the initialization
        self.system_prompt = system_prompt
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def __call__(self, message: str) -> str:
        """
        Process a user message and return the agent's response.

        This method allows the agent to be called like a function:
            response = agent("What movie should I watch?")

        Args:
            message: The user's input message

        Returns:
            str: The agent's response

        The method should:
        1. Append the user message to self.messages with role "user"
        2. Call self.execute() to get the LLM response
        3. Append the response to self.messages with role "assistant"
        4. Return the response
        """
        # TODO: Implement the call method
        self.messages.append({"role": "user", "content": message})
        response = self.execute()
        self.messages.append({"role": "assistant", "content": response})
        return response

    def execute(self) -> str:
        """
        Send the conversation to OpenAI and return the response.

        Returns:
            str: The content of the model's response

        The method should:
        1. Call client.chat.completions.create() with:
           - model="gpt-4o-mini"
           - temperature=0
           - messages=self.messages
        2. Return the response content from choices[0].message.content
        """
        # TODO: Implement the execute method
        chat_completion = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=self.messages
        )
        return chat_completion.choices[0].message.content

## Provided Code: Tools and Prompt

The following tools and system prompt are provided. Review them to understand how your agent class will be used.

In [5]:
# Movie database with runtime information
movie_database = {
    "action": {"title": "Mad Max: Fury Road", "director": "George Miller", "runtime": 120},
    "comedy": {"title": "The Grand Budapest Hotel", "director": "Wes Anderson", "runtime": 99},
    "drama": {"title": "The Shawshank Redemption", "director": "Frank Darabont", "runtime": 142},
    "horror": {"title": "Get Out", "director": "Jordan Peele", "runtime": 104},
    "sci-fi": {"title": "Inception", "director": "Christopher Nolan", "runtime": 148}
}


def get_movie_recommendation(genre: str) -> str:
    """
    Get a movie recommendation for a given genre.

    Args:
        genre: The movie genre (action, comedy, drama, horror, sci-fi)

    Returns:
        A string with the movie recommendation and runtime
    """
    genre = genre.lower().strip()
    if genre in movie_database:
        movie = movie_database[genre]
        return f"'{movie['title']}' directed by {movie['director']} ({movie['runtime']} minutes)"
    else:
        available = ", ".join(movie_database.keys())
        return f"Sorry, no recommendations for '{genre}'. Available genres: {available}"


def get_runtime_category(minutes: str) -> str:
    """
    Categorize a movie's runtime and suggest viewing context.

    Args:
        minutes: Runtime in minutes as a string

    Returns:
        A string describing the runtime category and viewing suggestion
    """
    try:
        runtime = int(minutes)
        if runtime < 100:
            return f"{runtime} minutes is a short film - perfect for a quick movie night."
        elif runtime < 130:
            return f"{runtime} minutes is a standard length - ideal for an evening watch."
        else:
            return f"{runtime} minutes is a longer film - make sure you have snacks ready!"
    except ValueError:
        return "Error: Please provide a valid number of minutes."


# Map action names to functions
available_actions = {
    "get_movie_recommendation": get_movie_recommendation,
    "get_runtime_category": get_runtime_category
}

print("Tools loaded successfully!")
print(f"Available actions: {list(available_actions.keys())}")

Tools loaded successfully!
Available actions: ['get_movie_recommendation', 'get_runtime_category']


In [6]:
# System prompt for the movie advisor agent
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer.

Use Thought to describe your reasoning about the question.
Use Action to run one of the available actions - then return PAUSE.
Observation will be the result of running those actions.
When you have enough information to answer the question, output your Answer.

Your available actions are:

get_movie_recommendation:
e.g. get_movie_recommendation: comedy
Returns a movie recommendation for the given genre.

get_runtime_category:
e.g. get_runtime_category: 120
Returns information about whether a movie runtime is short, standard, or long.

Example session:

Question: What's a good comedy movie?
Thought: The user wants a comedy movie recommendation. I should use the get_movie_recommendation action with the comedy genre.
Action: get_movie_recommendation: comedy
PAUSE

Observation: 'The Grand Budapest Hotel' directed by Wes Anderson (99 minutes)

Answer: I recommend 'The Grand Budapest Hotel' directed by Wes Anderson. It's a 99-minute comedy that's perfect for a fun movie night!
""".strip()

print("System prompt loaded!")

System prompt loaded!


## Test Your Implementation

Run the cells below to test your agent class structure.

In [7]:
# Test 1: Basic instantiation
print("Test 1: Creating agent with system prompt...")
test_agent = MovieAdvisorAgent(system_prompt)

# Verify the agent has the expected attributes
assert hasattr(test_agent, 'system_prompt'), "Agent should have system_prompt attribute"
assert hasattr(test_agent, 'messages'), "Agent should have messages attribute"
assert isinstance(test_agent.messages, list), "messages should be a list"
assert len(test_agent.messages) == 1, "messages should have one system message"
assert test_agent.messages[0]["role"] == "system", "First message should be system role"

print("PASSED: Agent created with correct structure")

Test 1: Creating agent with system prompt...
PASSED: Agent created with correct structure


In [8]:
# Test 2: Agent can be called and maintains conversation history
print("Test 2: Testing agent call and conversation history...")
print("=" * 60)

# Create a fresh agent
agent = MovieAdvisorAgent(system_prompt)

# Call the agent with a question
print("User: What's a good comedy movie?")
response = agent("What's a good comedy movie?")
print(f"\nAgent:\n{response}")

# Verify the agent returned a response and updated messages
assert response is not None, "Agent should return a response"
assert isinstance(response, str), "Response should be a string"
assert len(agent.messages) == 3, "Should have system + user + assistant messages"
assert agent.messages[1]["role"] == "user", "Second message should be user role"
assert agent.messages[1]["content"] == "What's a good comedy movie?", "User message content should match"
assert agent.messages[2]["role"] == "assistant", "Third message should be assistant role"
assert agent.messages[2]["content"] == response, "Assistant message should match returned response"

print("\n" + "=" * 60)
print("PASSED: Agent responds and maintains conversation history correctly!")

Test 2: Testing agent call and conversation history...
User: What's a good comedy movie?

Agent:
Thought: The user is looking for a recommendation for a comedy movie. I will use the get_movie_recommendation action with the comedy genre to find a suitable option.  
Action: get_movie_recommendation: comedy  
PAUSE  

PASSED: Agent responds and maintains conversation history correctly!


In [9]:
# Test 3: Single ReAct iteration with tool execution
# This query function runs ONE iteration: Agent responds -> Parse action -> Execute tool -> Show observation
# (The full loop that continues until Answer is the next exercise!)

action_re = re.compile(r'^Action: (\w+): (.*)$')  # Regex to parse action from response

def query(question):
    """Run a single ReAct iteration: get agent response, execute one tool, show observation."""
    bot = MovieAdvisorAgent(system_prompt)
    result = bot(question)
    print(result)

    # Parse the response for actions
    actions = [
        action_re.match(a)
        for a in result.split('\n')
        if action_re.match(a)
    ]

    if actions:
        action, action_input = actions[0].groups()
        if action not in available_actions:
            raise Exception(f"Unknown action: {action}: {action_input}")
        print(f" -- running {action}({action_input})")
        observation = available_actions[action](action_input)
        print("Observation:", observation)
    else:
        print("No action found in response")

print("Test 3: Running single ReAct iteration...")
print("=" * 60)
query("What's a good comedy movie?")
print("=" * 60)
print("PASSED: Single iteration completed!")

Test 3: Running single ReAct iteration...
Thought: The user is looking for a recommendation for a comedy movie. I will use the get_movie_recommendation action with the comedy genre to find a suitable option.  
Action: get_movie_recommendation: comedy  
PAUSE  
 -- running get_movie_recommendation(comedy  )
Observation: 'The Grand Budapest Hotel' directed by Wes Anderson (99 minutes)
PASSED: Single iteration completed!


## Success Criteria

Your implementation is correct if:

1. **Test 1 passes**: Agent initializes with system prompt stored and messages list containing the system message
2. **Test 2 passes**: Agent can be called with a message and:
   - Returns a string response from the LLM
   - Properly adds user message to conversation history
   - Properly adds assistant response to conversation history
3. **Test 3 runs**: A single ReAct iteration completes showing:
   - Agent responds with Thought/Action/PAUSE
   - Tool is executed and observation is printed

**Common issues to check:**
- If Test 1 fails, verify you're initializing `self.messages` as a list and adding the system message
- If Test 2 fails, ensure `__call__` adds messages with correct role keys and returns the response
- If the API call fails, check that `execute` uses the correct OpenAI API format

**Next steps:** In the next exercise, you'll implement the full agent loop that continues the cycle until the agent outputs an Answer!